In [ ]:

from typing import Any, Dict, List, Optional

from pydantic import BaseModel

from agente_avaliacao_imagens.schemas import AnaliseImagens, FeedbackImagens


class ReActInput(BaseModel):
    """Entrada do agente ReAct de análise de imagens."""

    fotos_urls: List[str] = []
    api_key: Optional[str] = None

c:\Users\jefer\Documents\Ciencia-de-dados\Preco-Imoveis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from phoenix.otel import register

tracer_provider = register(
  project_name="agente-react-imoveis",
  auto_instrument=True
)

08/05/2026 01:31:42 PM 📋 Ensuring phoenix working directory: C:\Users\jefer\.phoenix
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.schemas
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.tables
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.types
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.constraints
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.defaults
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.comments


OpenTelemetry Tracing Details
|  Phoenix Project: agente-react-imoveis
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/s/sehnemjeferson/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {'authorization': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



c:\Users\jefer\Documents\Ciencia-de-dados\Preco-Imoveis\.venv\Lib\site-packages\phoenix\otel\otel.py:433: UserWarning: Could not infer collector endpoint protocol, defaulting to HTTP.
  warnings.warn("Could not infer collector endpoint protocol, defaulting to HTTP.")


In [3]:
import logging
from typing import List

from langchain_core.tools import tool

from agente_avaliacao_imagens.prompts import PROMPT_DESCREVER_FOTO
from agente_avaliacao_imagens.utils import processar_todos_lotes

logger = logging.getLogger(__name__)


@tool
async def descrever_fotos(fotos_urls: List[str]) -> str:
    """Processa as fotos do imóvel e devolve a descrição técnica de cada imagem.

    Use esta ferramenta para obter a descrição das fotos. Depois, com base nela,
    preencha a análise estruturada final (scores, problemas, pontos fortes).

    Args:
        fotos_urls: lista de URLs das fotos do imóvel.
    """
    if not fotos_urls:
        return "Nenhuma URL de foto fornecida."
    
    logger.info(f"Processando {len(fotos_urls)} fotos para descrição.")

    descricao = await processar_todos_lotes(fotos_urls, 5, prompt=PROMPT_DESCREVER_FOTO)
    if not descricao:
        logger.error("Nao foi possivel descrever as fotos.")
        return "Falha ao descrever as fotos."
    return descricao

In [14]:
import json
import logging
import os
from typing import List, Optional

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langgraph.prebuilt import create_react_agent

from agente_avaliacao_imagens.schemas import AnaliseImagens
#from .tools import descrever_fotos

logger = logging.getLogger(__name__)

MODELO_AGENTE = os.getenv("MODELO_AGENTE_IMAGENS",
                          #"z-ai/glm-5.2"#V
                          "meta/llama-3.1-8b-instruct"V
                          #"nvidia/nemotron-3-ultra-550b-a55b" #V
                          # "google/gemma-4-31b-it" X
                          #"poolside/laguna-xs-2.1" #X
                          #"qwen/qwen-2.5-72b-instruct" #X
                          ) #"moonshotai/kimi-k2.6")"deepseek-ai/deepseek-v4-flash")

SYSTEM_PROMPT = """Você é um engenheiro civil e especialista em avaliação de imóveis para house flipping.

Sua tarefa é analisar as fotos de um imóvel e produzir um relatório técnico estruturado.

Passos:
1. Chame a ferramenta `descrever_fotos` com as URLs das fotos. Ela devolverá a descrição técnica de cada imagem.
2. Analise a descrição recebida e preencha a análise estruturada final com os campos abaixo.

Definição de cada campo do relatório final:

- **score_conservacao (float 0-10):** condição geral de conservação/mainutenção do que está visível (infiltrações, trincas, desgaste, estado de paredes/teto).
- **score_acabamento (float 0-10):** qualidade/padrão dos materiais (piso, revestimentos, metais, portas, esquadrias).
- **score_potencial_reforma (float 0-10):** o quanto é viável/vantajoso reformar o espaço (nota alta = boa estrutura que valoriza com melhorias; nota baixa = exige demolição pesada ou já está em ótimo estado).
- **confianca_imagem (float 0-10):** o quanto a descrição é confiável, clara e útil para uma avaliação técnica.
- **imagem_aceitavel (bool):** `true` se a foto mostra elementos reais do imóvel e é clara; `false` se for irrelevante (selfie, parede escura, objeto aleatório) ou a descrição for vaga demais.
- **problemas_visiveis (List[str]):** patologias, defeitos, danos ou sinais de desgaste identificados. Vazio se não houver.
- **pontos_fortes (List[str]):** aspectos positivos observados (iluminação natural, piso em bom estado, acabamento moderno, área espaçosa). Vazio se não houver.
- **observacoes (str):** resumo da opinião técnica. Se `imagem_aceitavel = false` ou as notas forem baixas, use este campo para justificar tecnicamente.

Regras importantes:
- Baseie-se APENAS na descrição fornecida pela ferramenta. Nunca invente ou infira o que não está visível.
- Se um aspecto não puder ser avaliado, pondere as notas de forma neutra e registre a limitação em `observacoes`.
- Se as fotos não retratarem um ambiente de imóvel (imagem irrelevante/ilegível), marque `imagem_aceitavel = false`, atribua `0.0` a todos os scores e explique em `observacoes`.
- Responda SEMPRE em português.
"""


def criar_agente_imagens(api_key: Optional[str] = None):
    model = ChatNVIDIA(
        model=MODELO_AGENTE,
        api_key=api_key or os.getenv("NVIDIA_API_KEY"),
    )
    return create_react_agent(
        model=model,
        tools=[descrever_fotos],
        prompt=SYSTEM_PROMPT,
        response_format=AnaliseImagens,
    )


async def analisar_imagens(
    fotos_urls: List[str],
    api_key: Optional[str] = None,
) -> AnaliseImagens:
    """Executa o agente ReAct de análise de imagens e devolve o relatório estruturado."""
    agente = criar_agente_imagens(api_key=api_key)

    mensagem_usuario = {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Analise as fotos do imóvel:\n"
                    + json.dumps(fotos_urls, ensure_ascii=False, indent=2)
                ),
            }
        ]
    }

    resultado = await agente.ainvoke(mensagem_usuario)
    resposta = resultado.get("structured_response")

    if isinstance(resposta, dict):
        return AnaliseImagens(**resposta)
    return resposta

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1975982450.py, line 16)

In [2]:
import pandas as pd
from pathlib import Path


cidade = 'joinville'

estado = 'sc'

BASE_DIR = Path.cwd().parent
PASTA_DADOS = BASE_DIR / 'dados' / cidade
df = pd.read_parquet(PASTA_DADOS / f'{cidade}_imoveis_limpo_2026-08.parquet')

In [ ]:
fotos = df['fotos'].iloc[6].tolist()

In [ ]:
fotos

['https://resizedimgs.zapimoveis.com.br/img/vr-listing/2be5cbec55da625c7063443f287d852b/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707',
 'https://resizedimgs.zapimoveis.com.br/img/vr-listing/80f6aae4cdef3bb2baf1c8609934408b/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707&seo=false',
 'https://resizedimgs.zapimoveis.com.br/img/vr-listing/532e18b9c870fd26cfc2a712304896ab/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707&seo=false',
 'https://resizedimgs.zapimoveis.com.br/img/vr-listing/ed7f7a3d250cc07765b1d97fa3e94448/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707&seo=false',
 'https://resizedimgs.zapimoveis.com.br/img/vr-listing/fa37d92847ba04c856f07049f497b1c3/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707&seo=false',
 'https://resizedimgs.zap

In [1]:
import pandas as pd
import json

In [1]:
#df = pd.read_json('olx_alugueis.json', lines=True)
import pandas as pd
from pathlib import Path


cidade = 'joinville'

estado = 'sc'

BASE_DIR = Path.cwd().parent
PASTA_DADOS = BASE_DIR / 'dados' / cidade
df_09 = pd.read_parquet(PASTA_DADOS / f'{cidade}_imoveis_limpo_2026-09.parquet')
df_08 = pd.read_parquet(PASTA_DADOS / f'{cidade}_imoveis_limpo_2026-08.parquet')

#with open(PASTA_DADOS/ 'joinville_aluguel_olx_2026-08.json', 'r', encoding='utf-8') as f:
#df = json.load(f)

# Verificando o urls que existiam no mes anterior

In [2]:
import pandas as pd
from collections import Counter

df_04 = pd.read_parquet(f'{PASTA_DADOS}/{cidade}_imoveis_limpo_2026-04.parquet')
df_05 = pd.read_parquet(f'{PASTA_DADOS}/{cidade}_imoveis_limpo_2026-05.parquet')
df_06 = pd.read_parquet(f'{PASTA_DADOS}/{cidade}_imoveis_limpo_2026-06.parquet')
df_07 = pd.read_parquet(f'{PASTA_DADOS}/{cidade}_imoveis_limpo_2026-07.parquet')
df_08 = pd.read_parquet(f'{PASTA_DADOS}/{cidade}_imoveis_limpo_2026-08.parquet')
df_09 = pd.read_parquet(f'{PASTA_DADOS}/{cidade}_imoveis_limpo_2026-09.parquet')

# --- Tamanho por mes ---
print('=== Registros por mes ===')
for i, df in enumerate([df_04, df_05, df_06, df_07, df_08, df_09], 4):
    print(f'  {i:02d}/2026: {len(df):>6} registros, {df.shape[1]} colunas')

# --- URLs por mes ---
urls = {}
for m, df in [(4, df_04), (5, df_05), (6, df_06), (7, df_07), (8, df_08), (9, df_09)]:
    urls[m] = set(df['url'].dropna())

# --- Intersecao com setembro ---
print('\n=== Intersecao com setembro (09) ===')
for m in [4, 5, 6, 7, 8]:
    inter = urls[m] & urls[9]
    so_antigo = urls[m] - urls[9]
    so_09 = urls[9] - urls[m]
    print(f'  {m:02d} vs 09: ambos={len(inter)}, so_no_{m:02d}={len(so_antigo)}, so_no_09={len(so_09)}')

# --- Novos e removidos por transicao ---
print('\n=== Novos por mes (nao existiam no mes anterior) ===')
for m in [5, 6, 7, 8, 9]:
    novos = urls[m] - urls[m - 1]
    print(f'  Novos em {m:02d}: {len(novos)}')

print('\n=== Removidos por mes (existiam no anterior) ===')
for m in [5, 6, 7, 8, 9]:
    removidos = urls[m - 1] - urls[m]
    print(f'  Removidos do {m - 1:02d} para {m:02d}: {len(removidos)}')

# --- Persistencia ---
todas_urls = []
for m in [4, 5, 6, 7, 8, 9]:
    todas_urls.extend(urls[m])

contagem = Counter(todas_urls)
persistencia = Counter(contagem.values())

print('\n=== Persistencia (em quantos meses cada URL aparece) ===')
for meses, qtd in sorted(persistencia.items()):
    print(f'  {meses} meses: {qtd} URLs')



=== Registros por mes ===
  04/2026:  34573 registros, 33 colunas
  05/2026:  26905 registros, 33 colunas
  06/2026:  24336 registros, 62 colunas
  07/2026:  23152 registros, 62 colunas
  08/2026:  25312 registros, 32 colunas
  09/2026:  24235 registros, 38 colunas

=== Intersecao com setembro (09) ===
  04 vs 09: ambos=5456, so_no_04=29117, so_no_09=18779
  05 vs 09: ambos=6349, so_no_05=20556, so_no_09=17886
  06 vs 09: ambos=6390, so_no_06=17946, so_no_09=17845
  07 vs 09: ambos=8019, so_no_07=15133, so_no_09=16216
  08 vs 09: ambos=8841, so_no_08=16471, so_no_09=15394

=== Novos por mes (nao existiam no mes anterior) ===
  Novos em 05: 6201
  Novos em 06: 12506
  Novos em 07: 8275
  Novos em 08: 11697
  Novos em 09: 15394

=== Removidos por mes (existiam no anterior) ===
  Removidos do 04 para 05: 13869
  Removidos do 05 para 06: 15075
  Removidos do 06 para 07: 9459
  Removidos do 07 para 08: 9537
  Removidos do 08 para 09: 16471

=== Persistencia (em quantos meses cada URL aparec

In [3]:
urls_comuns = urls[4] & urls[5] & urls[6] & urls[7] & urls[8] & urls[9]

df_filtrado = df_09[df_09['url'].isin(urls_comuns)]

print(f'URLs presentes em todos os 6 meses: {len(urls_comuns)}')
print(f'Registros em df_09 filtrados: {len(df_filtrado)}')

URLs presentes em todos os 6 meses: 3917
Registros em df_09 filtrados: 3917


In [4]:
df_filtrado[df_filtrado['tipo_imovel'].isin(['apartamento'])]['url'].to_dict()

{8404: 'https://www.chavesnamao.com.br/imovel/apartamento-a-venda-1-quarto-com-garagem-sc-joinville-gloria-45m2-RS261998/id-29140789/',
 8417: 'https://www.chavesnamao.com.br/imovel/apartamento-a-venda-2-quartos-com-garagem-sc-joinville-gloria-63m2-RS439204/id-39188957/',
 8422: 'https://www.chavesnamao.com.br/imovel/apartamento-a-venda-2-quartos-com-garagem-sc-joinville-centro-60m2-RS354538/id-41566765/',
 8423: 'https://www.chavesnamao.com.br/imovel/apartamento-a-venda-1-quarto-com-garagem-sc-joinville-itaum-60m2-RS304240/id-30197075/',
 8424: 'https://www.chavesnamao.com.br/imovel/apartamento-a-venda-2-quartos-com-garagem-sc-joinville-costa-e-silva-RS378000/id-32728378/',
 8429: 'https://www.chavesnamao.com.br/imovel/apartamento-a-venda-2-quartos-com-garagem-sc-joinville-vila-nova-65m2-RS277139/id-28699514/',
 8445: 'https://www.chavesnamao.com.br/imovel/apartamento-a-venda-2-quartos-com-garagem-sc-joinville-itaum-46m2-RS299900/id-32141384/',
 8448: 'https://www.chavesnamao.com.br/i

In [5]:
df_filtrado.groupby(['bairro', 'tipo_imovel'])['preco_por_m2'].aggregate(['mean', 'median','count']).head(50)

mean        median  count
bairro              tipo_imovel                                      
adhemar garcia      apartamento     5.092584e+03   5283.018868      7
                    casa            4.799988e+03   4607.843137     15
america             apartamento     9.843009e+03  10368.394737    425
                    casa            8.110012e+03   8140.703518     76
                    comercial       1.138116e+04  12238.380165      7
                    terreno         6.584261e+03   6984.924623      3
anita garibaldi     apartamento     8.818041e+03   8965.260756    322
                    casa            4.517152e+04   6887.196001     74
                    comercial       6.863553e+03   6863.553400      2
area rural          rural           6.758824e+03   6758.823529      1
atiradores          apartamento     1.015210e+04  10354.545455    171
                    casa            7.742876e+03   7348.059518     20
                    comercial       1.652374e+04  16523.744898      2
                    predio_inteiro  7.500000e+03   7500.000000      1
                    terreno         8.333333e+03   8333.333333      1
aventureiro         apartamento     5.455814e+03   5384.615385     17
                    casa            5.292796e+03   5319.148936     43
boa vista           apartamento     5.050468e+03   5314.516129      8
                    casa            6.563614e+03   5813.953488     55
                    comercial       4.072333e+03   4072.333152      2
                    outros          5.428571e+03   5428.571429      1
boehmerwald         apartamento     5.477753e+03   5674.242424      6
                    casa            4.452675e+03   4380.952381     21
bom retiro          apartamento     7.386882e+03   7089.552239     76
                    casa            6.258183e+03   5973.913043    103
                    comercial       5.106383e+03   5106.382979      1
bucarein            apartamento     7.778603e+03   8490.916667     45
                    casa            6.005950e+03   6296.296296     21
                    comercial       7.812500e+03   7812.500000      1
centro              apartamento     9.660379e+03  10000.000000    135
                    casa            7.508658e+03   6757.099794      6
                    comercial                inf   6250.000000     21
comasa              apartamento     5.035716e+03   5204.681589      6
                    casa            4.983222e+03   4990.196078     23
costa e silva       apartamento              inf   7047.737069    224
                    casa            1.111278e+04   5750.000000    140
                    comercial       7.290684e+03   7252.251414      4
                    terreno         8.771930e+03   8771.929825      1
distrito industrial apartamento     6.200993e+03   6200.992949      2
                    casa            6.265238e+03   6265.238095      2
espinheiros         apartamento     7.091591e+03   6865.384615      5
                    casa            4.941478e+03   4848.450057     22
                    terreno         1.624103e+03   1650.000000      3
fatima              apartamento     6.144822e+03   5978.019324      8
                    casa            3.992428e+03   4329.670330      8
                    terreno         4.333333e+03   4333.333333      1
floresta            apartamento     6.875037e+03   7031.250000     47
                    casa            5.597898e+03   5390.625000    111
                    terreno         1.010101e+03   1010.101010      1
gloria              apartamento     8.938219e+03   8478.411448    106

In [6]:
import pandas as pd
import numpy as np

# --- Filtrar URLs presentes em todos os 6 meses ---
urls_comuns = urls[4] & urls[5] & urls[6] & urls[7] & urls[8] & urls[9]
df_filtrado = df_09[df_09['url'].isin(urls_comuns)].copy()

print(f'URLs presentes em todos os 6 meses: {len(urls_comuns)}')
print(f'Registros em df_09 filtrados: {len(df_filtrado)}')
print()

# --- 1. Visao geral por tipo ---
print('=== Distribuicao por tipo de imovel ===')
tipo_resumo = df_filtrado.groupby('tipo_imovel').agg(
    qtd=('url', 'count'),
    preco_m2_medio=('preco_por_m2', 'mean'),
    preco_m2_mediana=('preco_por_m2', 'median'),
    metragem_media=('metragem', 'mean'),
    valor_medio=('valor_imovel', 'mean'),
).round(0)
print(tipo_resumo.to_string())
print()

# --- 2. Top bairros por preco m2 ---
print('=== Top 15 bairros por preco m2 medio ===')
bairro_resumo = df_filtrado.groupby('bairro').agg(
    qtd=('url', 'count'),
    preco_m2_medio=('preco_por_m2', 'mean'),
    preco_m2_mediana=('preco_por_m2', 'median'),
    metragem_media=('metragem', 'mean'),
    valor_medio=('valor_imovel', 'mean'),
).round(0)
bairro_resumo = bairro_resumo[bairro_resumo['qtd'] >= 5].sort_values('preco_m2_medio', ascending=False)
print(bairro_resumo.head(15).to_string())
print()

# --- 3. Cruzamento bairro x tipo (preco m2 medio) ---
print('=== Preco m2 medio: bairro x tipo (top 30) ===')
cruzado = df_filtrado.groupby(['bairro', 'tipo_imovel']).agg(
    qtd=('url', 'count'),
    preco_m2_medio=('preco_por_m2', 'mean'),
    preco_m2_mediana=('preco_por_m2', 'median'),
).round(0)
cruzado = cruzado[cruzado['qtd'] >= 3].sort_values('preco_m2_medio', ascending=False)
print(cruzado.head(30).to_string())
print()

# --- 4. Faixas de preco por bairro ---
print('=== Distribuicao por faixa de preco por bairro (top 10 bairros) ===')
top_bairros = df_filtrado['bairro'].value_counts().head(10).index
faixa_bairro = df_filtrado[df_filtrado['bairro'].isin(top_bairros)].groupby(['bairro', 'faixa']).size().unstack(fill_value=0)
print(faixa_bairro.to_string())
print()

# --- 5. Metricas por quartos ---
print('=== Preco m2 por quantidade de quartos ===')
quartos_resumo = df_filtrado.groupby('quartos').agg(
    qtd=('url', 'count'),
    preco_m2_medio=('preco_por_m2', 'mean'),
    preco_m2_mediana=('preco_por_m2', 'median'),
    metragem_media=('metragem', 'mean'),
    valor_medio=('valor_imovel', 'mean'),
).round(0)
print(quartos_resumo.to_string())
print()

# --- 6. Fonte dos dados ---
print('=== Distribuicao por fonte ===')
print(df_filtrado['fonte'].value_counts().to_string())

URLs presentes em todos os 6 meses: 3917
Registros em df_09 filtrados: 3917

=== Distribuicao por tipo de imovel ===
                 qtd  preco_m2_medio  preco_m2_mediana  metragem_media  valor_medio
tipo_imovel                                                                        
apartamento     2159             inf            8248.0           110.0     969671.0
casa            1670             inf            5909.0           142.0     879145.0
comercial         42             inf            7596.0            96.0     791233.0
galpao             4          4441.0            3954.0           198.0     851250.0
outros             1          5429.0            5429.0            70.0     380000.0
predio_inteiro     1          7500.0            7500.0           200.0    1500000.0
rural             12         26420.0           11667.0           152.0    2640833.0
terreno           28             inf            3759.0           210.0    1293941.0

=== Top 15 bairros por preco m2 medio ===


# testes

In [25]:
#df = pd.read_parquet(f'{PASTA_DADOS}/joinville_chave_mao_2026-09.parquet')
df = pd.read_parquet(f'{PASTA_DADOS}/joinville_imovelweb_2026-09.parquet')

In [26]:
df.shape

(10169, 25)

In [27]:
df.isna().sum()

url                             0
titulo                          2
metragem                      255
metragem_total                267
metragem_util                 797
quartos                      1361
suites                       3709
banheiros                    1299
vagas                        1470
idade                        3648
valor_imovel                  102
condominio                   7417
endereco                       26
bairro                         26
cidade                         26
uf                             33
descricao                      26
data_criacao                   26
caracteristicas                 0
caracteristicas_privativa       0
caracteristicas_comum           0
fotos                           0
lat                          4518
lng                          4518
fonte                           0
dtype: int64

In [9]:
df['idade'].value_counts(dropna=False)

idade
None                3648
Breve Lançamento    2038
2                    905
1                    872
5                    436
3                    377
4                    246
16                   127
12                   105
20                   101
10                    99
31                    92
6                     90
11                    88
14                    87
30                    85
7                     72
9                     66
26                    60
15                    54
28                    47
33                    42
8                     37
13                    35
17                    31
25                    31
21                    27
40                    26
32                    26
44                    22
24                    22
23                    20
46                    17
18                    17
22                    15
29                    15
48                    15
41                    14
43                    14
34                 

In [23]:
df = pd.read_parquet(f'{PASTA_DADOS}/joinville_zap_2026-09_0_50.parquet')

In [24]:
df

,url,titulo,metragem,banheiros,vagas,quartos,valor_imovel,condominio,endereco,iptu,descricao,data_criacao,caracteristicas,caracteristicas_privativa,caracteristicas_comum,fotos,link_maps
0,https://www.zapimoveis.com.br/imovel/venda-apa...,"Apartamento com 2 Quartos à venda, 50m² - Flor...",50 m²,1,1,2,None,R$ 300/mês,"Avenida Antônio Ramos Alvim - Floresta, Joinvi...",R$ 330,"Apartamento com 50,50 m² de área total, ideal ...","Anúncio criado em 2 de setembro de 2026, atual...","[50 m², 2 quartos, 1 banheiro, 1 vaga, Armário...",[Armário na cozinha],"[Portaria 24h, Espaço gourmet, Bicicletário, C...",[https://resizedimgs.zapimoveis.com.br/img/vr-...,https://www.google.com/maps/embed/v1/place?key...
1,https://www.zapimoveis.com.br/imovel/venda-apa...,"Apartamento com 2 Quartos à venda, 49m² - Guan...",49 m²,1,1,2,None,R$ 200/mês,"Rua Graciosa, 720 - Guanabara, Joinville - SC",R$ 46,Localização: Em uma excelente localização de J...,"Anúncio criado em 12 de junho de 2026, atualiz...","[49 m², 2 quartos, 1 banheiro, 1 vaga, 2º anda...","[2º andar, Área de serviço, Cozinha]","[Churrasqueira, Condomínio fechado]",[https://resizedimgs.zapimoveis.com.br/img/vr-...,https://www.google.com/maps/embed/v1/place?key...
2,https://www.zapimoveis.com.br/imovel/venda-apa...,"Apartamento com 2 Quartos à venda, 45m² - João...",45 m²,1,1,2,None,Isento,"João Costa, Joinville - SC",Não informado,APARTAMENTO COM 2 DORMITÓRIOS À VENDA NO BAIRR...,"Anúncio criado em 7 de julho de 2026, atualiza...","[45 m², 2 quartos, 1 banheiro, 1 vaga, 2º anda...","[2º andar, Varanda, Armário na cozinha, Armári...","[Salão de festas, Quadra poliesportiva, Ronda/...",[https://resizedimgs.zapimoveis.com.br/img/vr-...,https://www.google.com/maps/embed/v1/place?key...
3,https://www.zapimoveis.com.br/imovel/venda-apa...,"Apartamento com 2 Quartos à venda, 50m² - Cost...",50 m²,2,1,2,None,Não informado,"Costa E Silva, Joinville - SC",Não informado,"Excelente lançamento no bairro Costa e Silva, ...","Anúncio criado em 13 de agosto de 2025, atuali...","[50 m², 2 quartos, 2 banheiros, 1 vaga, 1 suít...",[Cozinha],"[Piscina, Salão de festas, Churrasqueira, Play...",[https://resizedimgs.zapimoveis.com.br/img/vr-...,https://www.google.com/maps/embed/v1/place?key...
4,https://www.zapimoveis.com.br/imovel/venda-apa...,"Apartamento com 2 Quartos à venda, 40m² - Vila...",40 m²,1,1,2,None,R$ 300/mês,"Rua Harold Carlos Miers, 602 - Vila Nova, Join...",R$ 250,Apartamento com 2 quartos em condomínio comple...,"Anúncio criado em 23 de outubro de 2025, atual...","[40 m², 2 quartos, 1 banheiro, 1 vaga, 0 suíte...",[],[],[https://resizedimgs.zapimoveis.com.br/img/vr-...,https://www.google.com/maps/embed/v1/place?key...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,https://www.zapimoveis.com.br/imovel/venda-cas...,"Casa com 2 Quartos à venda, 50m² - Aventureiro",50 m²,1,2,2,None,R$ 1/mês,"Rua Carlos Roberto Vilpert - Aventureiro, Join...",R$ 100,Casa a venda em Rua Carlos Roberto Vilpert - A...,"Anúncio criado em 7 de setembro de 2026, atual...","[50 m², 2 quartos, 1 banheiro, 2 vagas, 0 suít...",[Aceita animais],[],[https://resizedimgs.zapimoveis.com.br/img/vr-...,https://www.google.com/maps/embed/v1/place?key...
96,https://www.zapimoveis.com.br/imovel/venda-apa...,"Apartamento com 2 Quartos à venda, 44m² - Vila...",44 m²,1,1,2,None,Não informado,"Rua Guilherme Kurtz, 140 - Vila Nova, Joinvill...",Não informado,"Descubra este apartamento em Vila Nova, Joinvi...","Anúncio criado em 23 de junho de 2026, atualiz...","[44 m², 2 quartos, 1 banheiro, 1 vaga, 0 suítes]",[],[],[https://resizedimgs.zapimoveis.com.br/img/vr-...,https://www.google.com/maps/embed/v1/place?key...
97,https://www.zapimoveis.com.br/imovel/venda-loj...,Loja / Salão / Ponto Comercial com 1 Quarto à ...,38 m²,1,--,1,None,Não informado,"América, Joinville - SC",Não informado,Apartamento Piscina América Joinville Apartame...,"Anúncio criado em 10 de agosto de 2026, atuali...","[38 m², 1 quarto, 1 banh

In [17]:
df[df['fonte'].isin(['olx'])].loc[:,['metragem', 'metragem_total', 'metragem_util']]

,metragem,metragem_total,metragem_util
17299,61.0,None,None
17300,40.0,None,None
17301,65.0,None,None
17302,51.0,None,None
17303,46.0,None,None
...,...,...,...
22660,107.0,None,None
22661,124.0,None,None
22662,109.0,None,None
22663,92.0,None,None


In [ ]:
df.loc[:,['metragem', 'metragem_total', 'metragem_util', 'fonte']]

,metragem,metragem_total,metragem_util,fonte
0,176.0,176 m²,176 m²,chave_mao
1,248.0,248 m²,178 m²,chave_mao
2,200.0,None,200 m²,chave_mao
3,199.0,199 m²,199 m²,chave_mao
4,177.0,None,177 m²,chave_mao
...,...,...,...,...
21720,91.0,None,None,imovelweb
21721,93.0,None,None,imovelweb
21722,100.0,None,None,imovelweb
21723,94.0,None,None,imovelweb
